# Lost in the Museum - contrastive fine-tuning, corrected augmentation

### What went wrong the first time

A previous fine-tuning run scored **0.68456**, against **0.775** for the same
backbone and resolution zero-shot. Fine-tuning did not merely fail to help, it
damaged the representation.

The cause was the augmentation, not the method. That pipeline was reverse-
engineered from a single query image and was far too aggressive: it cropped away
up to 45% of the painting and downscaled to roughly 64 pixels. Teaching a model
to survive distortions that never occur means discarding exactly the fine detail
that separates two similar paintings -- and this gallery contains 9,000 artwork
distractors, including different works by the same artist on the same subject.

The offline check missed it because it evaluated on the same augmentation family
used for training. Varying the *strength* looked like a held-out test, but the
axis that mattered was the *kind* of distortion.

### What changed

Real queries were located by their padding signature (17932, 11728, 05362,
16000) and inspected directly. They are far milder than assumed:

| | Real queries | Old augmentation | Corrected |
|---|---|---|---|
| Rotation + pale padding | ~5-10 deg | up to 12 deg | 3-10 deg, magnitude biased away from zero |
| Cropping | none, whole painting visible | up to 45% removed | none |
| Downscale | mild | to ~64 px | 0.45-1.0x |
| Blur | variable, often sharp | always | 50% of the time, light |

Training is also gentler -- two epochs, fewer unfrozen blocks -- so the
pretrained representation is nudged rather than overwritten.

**Settings:** GPU ON, Internet ON. Roughly 1.5-2 hours.


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None


def find_data_dir():
    best, best_n = None, 0
    for d in Path("/kaggle/input").rglob("*"):
        if d.is_dir():
            n = sum(1 for _ in d.glob("*.png"))
            if n > best_n:
                best, best_n = d, n
    return best, best_n


DATA_DIR, _n = find_data_dir()
print("Data:", DATA_DIR, f"({_n} png)")
assert DATA_DIR is not None and _n == 20000, "expected 20000 images"

BACKBONE    = "dinov2_vitl14"
SIZE        = 518          # matches the best zero-shot configuration
BATCH       = 6
EPOCHS      = 2            # gentle: nudge the representation, do not retrain it
LR, HEAD_LR = 5e-6, 5e-4   # lower than the failed run
TEMPERATURE = 0.05
TRAINABLE   = 2            # fewer unfrozen blocks than the failed run (was 4)
PCA_DIM     = 1536
SEED        = 0

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Enable Settings -> Accelerator -> GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"VRAM {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")
torch.manual_seed(SEED); np.random.seed(SEED)

## Corrected degradation

`expand=True` on the rotation is the detail that reproduces the real queries:
without it the rotation crops the corners off instead of padding them, and the
pale triangles that every real query shows never appear.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
PAD_FILL = 240


class RotatePad:
    """Rotate with expand=True so the full painting survives inside a pale frame.

    `expand=True` is the detail that reproduces the look of the real queries:
    without it the rotation crops the corners off instead of padding them.
    """

    def __init__(self, max_deg: float = 10.0, fill: int = PAD_FILL):
        self.max_deg, self.fill = max_deg, fill

    def __call__(self, img: Image.Image) -> Image.Image:
        # Sample the *magnitude* away from zero: a uniform(-10, 10) draw spends
        # most of its mass on 1-2 degree tilts that leave almost no padding,
        # whereas every real query inspected showed a clearly visible rotation.
        mag = float(np.random.uniform(0.3 * self.max_deg, self.max_deg))
        deg = mag if np.random.rand() < 0.5 else -mag
        return img.rotate(deg, resample=Image.BICUBIC, expand=True,
                          fillcolor=(self.fill,) * 3)


class MildDownscale:
    """Lose some resolution, but nothing like the original 64px collapse."""

    def __init__(self, lo: float = 0.45, hi: float = 1.0):
        self.lo, self.hi = lo, hi

    def __call__(self, img: Image.Image) -> Image.Image:
        f = float(np.random.uniform(self.lo, self.hi))
        if f > 0.97:
            return img
        w, h = img.size
        small = (max(32, int(w * f)), max(32, int(h * f)))
        return img.resize(small, Image.BILINEAR).resize((w, h), Image.BICUBIC)


def query_view(size: int, strength: float = 1.0) -> transforms.Compose:
    return transforms.Compose([
        RotatePad(max_deg=10.0 * strength),
        transforms.RandomPerspective(distortion_scale=0.10 * strength, p=0.35,
                                     fill=PAD_FILL),
        # Resize the whole rotated frame rather than cropping into it. Cropping
        # here would remove the pale corners, which are the single most visible
        # signature of a real query -- the model must learn to look past them,
        # not be shielded from them.
        transforms.Resize((size, size),
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ColorJitter(brightness=0.25 * strength, contrast=0.25 * strength,
                               saturation=0.35 * strength, hue=0.02 * strength),
        transforms.RandomGrayscale(p=0.10),
        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=5, sigma=(0.3, 1.2 * strength))], p=0.5),
        MildDownscale(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def gallery_view(size: int, jitter: bool = True) -> transforms.Compose:
    steps = [transforms.Resize((size, size),
                               interpolation=transforms.InterpolationMode.BICUBIC)]
    if jitter:
        steps.append(transforms.ColorJitter(brightness=0.06, contrast=0.06))
    steps += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(steps)

## Dataset and model

Only the last two blocks train. The projection head is used for the contrastive
loss and discarded at inference -- the representation just beneath it transfers
better to retrieval.


In [ ]:
class PairDataset(Dataset):
    """Clean anchor plus one degraded positive per image."""

    def __init__(self, paths, size, strength=1.0):
        self.paths, self.size = paths, size
        self.anchor_tf = gallery_view(size)
        self.query_tf = query_view(size, strength)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception as e:
            print(f"  ! unreadable {self.paths[i].name}: {e}", flush=True)
            blank = torch.zeros(3, self.size, self.size)
            return blank, blank, i
        return self.anchor_tf(img), self.query_tf(img), i


class InferenceDataset(Dataset):
    def __init__(self, paths, size):
        self.paths, self.size = paths, size
        self.tf = gallery_view(size, jitter=False)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.paths[i]).convert("RGB")), i
        except Exception:
            return torch.zeros(3, self.size, self.size), i   # never drop a row


def gem_pool(patch_tokens, p=3.0, eps=1e-6):
    return patch_tokens.clamp(min=eps).pow(p).mean(dim=1).pow(1.0 / p)


class RetrievalNet(nn.Module):
    def __init__(self, backbone_name, trainable_blocks, proj_dim=512):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, verbose=False)
        d = self.backbone.embed_dim
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-trainable_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True
        self.head = nn.Sequential(nn.Linear(2 * d, 2 * d), nn.GELU(), nn.Linear(2 * d, proj_dim))

    def features(self, x):
        o = self.backbone.forward_features(x)
        return torch.cat([F.normalize(o["x_norm_clstoken"], dim=1),
                          F.normalize(gem_pool(o["x_norm_patchtokens"]), dim=1)], dim=1)

    def forward(self, x):
        return F.normalize(self.head(self.features(x)), dim=1)

    def trainable_parameters(self):
        return [p for p in self.parameters() if p.requires_grad]

## Train


In [ ]:
def info_nce(a, b, t):
    logits = a @ b.t() / t
    lab = torch.arange(len(a), device=a.device)
    return 0.5 * (F.cross_entropy(logits, lab) + F.cross_entropy(logits.t(), lab))


paths = sorted(DATA_DIR.glob("*.png"))
model = RetrievalNet(BACKBONE, TRAINABLE).to(device)
print(f"trainable {sum(p.numel() for p in model.trainable_parameters())/1e6:.1f}M / "
      f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M")

head_ids = {id(p) for p in model.head.parameters()}
optim = torch.optim.AdamW([
    {"params": [p for p in model.trainable_parameters() if id(p) not in head_ids], "lr": LR},
    {"params": list(model.head.parameters()), "lr": HEAD_LR},
], weight_decay=0.05)

loader = DataLoader(PairDataset(paths, SIZE), batch_size=BATCH, shuffle=True,
                    num_workers=2, drop_last=True, pin_memory=True)
steps_total = EPOCHS * len(loader)
sched = torch.optim.lr_scheduler.OneCycleLR(optim, max_lr=[LR, HEAD_LR],
                                            total_steps=steps_total, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")

t0, step = time.time(), 0
for epoch in range(EPOCHS):
    model.train(); running = seen = 0
    for anchor, positive, _ in loader:
        anchor = anchor.to(device, non_blocking=True)
        positive = positive.to(device, non_blocking=True)
        optim.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = info_nce(model(anchor), model(positive), TEMPERATURE)
        scaler.scale(loss).backward(); scaler.step(optim); scaler.update(); sched.step()
        running += loss.item() * len(anchor); seen += len(anchor); step += 1
        if step % 200 == 0:
            r = step / (time.time() - t0)
            print(f"  epoch {epoch} step {step}/{steps_total}  loss {running/seen:.4f}  "
                  f"{r:.2f} it/s  ETA {(steps_total-step)/r/60:.0f} min", flush=True)
    torch.save({"model": model.state_dict(), "epoch": epoch}, "/kaggle/working/finetuned.pt")
    print(f"epoch {epoch}: loss {running/seen:.4f}", flush=True)
print(f"trained in {(time.time()-t0)/60:.1f} min")

## Embed and submit

Whitened PCA, unchanged -- it was the largest single lever in the zero-shot phase.


In [ ]:
def l2(x, eps=1e-12):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


loader = DataLoader(InferenceDataset(paths, SIZE), batch_size=BATCH * 2,
                    shuffle=False, num_workers=2, pin_memory=True)
feats, t0 = None, time.time()
model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    for imgs, idxs in loader:
        v = model.features(imgs.to(device, non_blocking=True)).float().cpu().numpy()
        if feats is None:
            feats = np.zeros((len(paths), v.shape[1]), dtype=np.float32)
        feats[idxs.numpy()] = v
print(f"embedded {feats.shape} in {(time.time()-t0)/60:.1f} min")
np.save("/kaggle/working/features_ft2.npy", feats)

x = l2(feats.astype(np.float64))
mu = x.mean(axis=0, keepdims=True)
_, s, vt = np.linalg.svd(x - mu, full_matrices=False)
dim = min(PCA_DIM, x.shape[1])
x = l2((x - mu) @ vt[:dim].T / (s[:dim] / np.sqrt(len(x) - 1) + 1e-8)).astype(np.float32)
print(f"PCA {feats.shape[1]} -> {dim}  explains {(s[:dim]**2).sum()/(s**2).sum():.1%}")

df = pd.DataFrame(x, columns=[f"feature_{i}" for i in range(x.shape[1])])
df.insert(0, "image_name", [p.name for p in paths])
df["ID"] = df["image_name"]
df.to_csv("/kaggle/working/submission.csv", index=False, float_format="%.6f")
print(f"rows={len(df)}  cols={df.shape[1]}")
df.iloc[:3, :5]